In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
from prophet import TesterModel

In [3]:
# the TesterModel requires a specific data format
# this is specified in the config.yaml file

data = {
    'icd':pl.read_parquet('/home/niels/Desktop/prophet/tests/data/icd.parquet').select(
        pl.col('bdsp_patient_id').cast(pl.UInt32).alias('id'),
        pl.col('icd'),
        pl.col('date_icd').cast(pl.Date).alias('date')
    ),
    'note':pl.read_parquet('/home/niels/Desktop/prophet/tests/data/note.parquet').select(
        pl.col('note'),
        pl.col('source'),
        pl.col('val').alias('annot'),
        pl.col('date'),
        pl.col('BDSPPatientID').cast(pl.UInt32).alias('id'),
    ),
}


In [4]:
config_path = '/home/niels/Desktop/prophet/src/prophet/models/template/config.yaml'

In [5]:
model = TesterModel(config_path)

[nltk_data] Downloading package punkt to /home/niels/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/niels/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
features = model.preprocess(data, show_progress=True)

2025-05-13 13:24:49,775	INFO worker.py:1841 -- Started a local Ray instance.


(pid=3247578) Processing notes 0:   0%|          | 0.00/3.17k [00:00<?, ?it/s]

In [8]:
# Adding the Y values and the hospital source to the feature matrix for model training

final_annot = data['note'].with_columns(
    pl.col('source').str.split('_').list.first().alias('source')
).group_by(['date', 'id']).agg(
    pl.col('annot').max().alias('annot'),
    pl.col('source').first()
)
features = features.join(
    final_annot,
    on=['date', 'id'],
    how='left'
)
features = features.filter(
    (pl.col('annot') == 1) | (pl.col('annot') == 0)
)

In [12]:
results, curves_data, final_model, best_params, best_model_name = model.train_model(
    features, 
    target_column='annot', 
    output_dir='/home/niels/Desktop/prophet/tests/output_data',
    source_column='source'
)

Found 2 unique sources: ['bidmc', 'mgh']

===== Evaluating LogisticRegression =====
Detected 284 binary features and 3 continuous features

Training LogisticRegression on source: bidmc
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will

Best parameters for LogisticRegression on bidmc: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 2000, 'model__penalty': 'l2', 'model__random_state': 42, 'model__solver': 'lbfgs'}
Testing LogisticRegression on source: mgh
LogisticRegression bidmc → mgh Performance:
Accuracy: 0.4603
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
ROC AUC Score: 0.8344
AUPRC Score: 0.8531
Detected 284 binary features and 3 continuous features

Training LogisticRegression on source: mgh
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/versions/3.11.11/envs/prophet/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/home/niels/.pyenv/v

Best parameters for LogisticRegression on mgh: {'model__C': 0.1, 'model__class_weight': None, 'model__max_iter': 2000, 'model__penalty': 'l2', 'model__random_state': 42, 'model__solver': 'lbfgs'}
Testing LogisticRegression on source: bidmc
LogisticRegression mgh → bidmc Performance:
Accuracy: 0.8105
Precision: 0.4527
Recall: 0.6498
F1 Score: 0.5337
ROC AUC Score: 0.8146
AUPRC Score: 0.5523

===== Evaluating LogisticRegression_Liblinear =====
Detected 284 binary features and 3 continuous features

Training LogisticRegression_Liblinear on source: bidmc
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters for LogisticRegression_Liblinear on bidmc: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 2000, 'model__penalty': 'l1', 'model__random_state': 42, 'model__solver': 'liblinear'}
Testing LogisticRegression_Liblinear on source: mgh
LogisticRegression_Liblinear bidmc → mgh Performance:
Accuracy: 0.7997
Precision: 0.8117
Recall: 0.8189
F1 Score

In [13]:
results

model,train_source,test_source,accuracy,precision,recall,f1,roc_auc,auprc,best_params
str,str,str,f64,f64,f64,f64,f64,f64,str
"""LogisticRegression""","""bidmc""","""mgh""",0.460254,0.0,0.0,0.0,0.834405,0.853114,"""{'model__C': 0.1, 'model__clas…"
"""LogisticRegression""","""mgh""","""bidmc""",0.810526,0.452747,0.649842,0.533679,0.814635,0.552329,"""{'model__C': 0.1, 'model__clas…"
"""LogisticRegression_Liblinear""","""bidmc""","""mgh""",0.799682,0.811679,0.818851,0.815249,0.873465,0.868258,"""{'model__C': 0.1, 'model__clas…"
"""LogisticRegression_Liblinear""","""mgh""","""bidmc""",0.807368,0.446855,0.649842,0.529563,0.815415,0.551839,"""{'model__C': 0.1, 'model__clas…"
"""RandomForest""","""bidmc""","""mgh""",0.807631,0.785621,0.885125,0.83241,0.861396,0.83166,"""{'model__class_weight': 'balan…"
…,…,…,…,…,…,…,…,…,…
"""GradientBoosting""","""mgh""","""bidmc""",0.854737,0.584362,0.44795,0.507143,0.808551,0.522305,"""{'model__learning_rate': 0.1, …"
"""XGBoost""","""bidmc""","""mgh""",0.699523,0.800399,0.590574,0.679661,0.843604,0.809283,"""{'model__colsample_bytree': 0.…"
"""XGBoost""","""mgh""","""bidmc""",0.874211,0.683962,0.457413,0.548204,0.899467,0.638204,"""{'model__colsample_bytree': 1.…"
